# Chapitre 1 : Introduction à la Data Science appliquée

*MLSMM2151 Data Science · Séance 1, vendredi 13 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 1 sur 1 : de Data Analytics à la Data Science

**Durée estimée : 45 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- retrouver, pour chaque concept vu en Data Analytics, le chapitre où ce cours
  le met en pratique ;
- charger un jeu de données réel et en tirer trois observations en cinq minutes ;
- reconnaître l'étape où un projet est en train d'échouer ;
- dire, face à une demande métier, de quel paradigme elle relève, et s'il
  s'agit d'une régression ou d'une classification ;
- énoncer les deux questions métier qui structurent tout le cours.

## 1.1. De Data Analytics à la Data Science

Vous avez suivi [Data Analytics (MLSMM2116)](https://uclouvain.be/cours-2026-MLSMM2116) ce quadrimestre. Vous y avez vu ce qu'est la data
science, le cycle de vie d'un projet, les familles d'analyse (descriptive,
diagnostique, prédictive, prescriptive), l'apprentissage supervisé et non
supervisé, le sur-apprentissage, le découpage entraînement, validation et test,
les métriques de classification, le clustering, et vous avez produit un
reporting sous Power BI.

**Ce cours ne refait pas cette théorie. Il la met en pratique**, en Python, sur
des données réelles, avec des devoirs rendus dans git. Trois choses changent :

- **vous écrivez le code.** Chaque méthode est exécutée, modifiée, comparée,
  dans un notebook ;
- **les données sont sales.** Pas de tableau propre sur une slide : un fichier
  de 6 721 annonces avec des prix en texte et des colonnes vides ;
- **le travail doit être reproductible.** Un résultat que personne ne peut
  relancer n'existe pas. C'est l'objet du chapitre 2, et de chaque devoir.

Voici où ce cours reprend ce que vous connaissez :

| Vu en Data Analytics | Ce que ce cours en fait | Chapitre |
|---|---|---|
| Cycle de vie d'un projet | La structure de chaque devoir, du cadrage à la conclusion | 1, devoirs |
| Problèmes liés aux données, nettoyage, ETL | Diagnostiquer et nettoyer soi-même, en pandas | 3 |
| Standardisation, extraction de variables | Encodage, mise à l'échelle, pipelines scikit-learn | 4 |
| Supervisé et non supervisé, sur-apprentissage, train/validation/test | Les faire, et voir la fuite de données | 5 |
| Régression, classification | Modèles linéaires, logistiques, arbres, en code | 6 |
| Métriques de classification | Les calculer, les choisir, et voir quand elles mentent | 7 |
| Découpage, rééquilibrage des classes | Validation croisée, hyperparamètres, déséquilibre | 8 |
| KNN, SVM, réseaux de neurones, cités | Les entraîner et les comparer | 9, 10 |
| Clustering, K-Means | K-Means en pratique, DBSCAN, évaluer une segmentation | 11 |
| Réduction de dimension | La PCA, en pratique et dans un pipeline | 12 |

### Le cycle de vie, rappel en une table

Vous le connaissez. Ici, il devient la structure attendue de tout travail
rendu : chaque devoir le parcourt en entier.

| # | Étape | La question à laquelle elle répond |
|---|---|---|
| 1 | **Compréhension du problème métier** | Quelle décision va changer grâce à ce travail ? |
| 2 | **Exploration des données** | Qu'est-ce que j'ai vraiment sous la main ? |
| 3 | **Préparation des données** | Comment rendre ces données exploitables sans les trahir ? |
| 4 | **Construction de modèles** | Quel modèle capture le signal utile ? |
| 5 | **Optimisation et validation** | Ce résultat tiendra-t-il sur des données jamais vues ? |
| 6 | **Communication des résultats** | Qu'est-ce que mon interlocuteur doit retenir et décider ? |

Deux remarques, que la pratique rendra concrètes. **Ce n'est pas linéaire** :
l'exploration révèle que la question était mal posée, et on repart à l'étape 1.
**L'étape 1 est celle qu'on saute, et c'est celle qui coûte le plus cher** : un
modèle parfait qui répond à la mauvaise question vaut exactement zéro. La
section 1.3 en donne un exemple.

## 1.2. Premier contact avec les données du cours

Tout le cours utilise le même jeu de données, pour que vous n'ayez à comprendre
qu'un seul contexte métier et que vous puissiez comparer ce que chaque méthode
apporte.

**Le contexte.** Vous travaillez pour une société qui gère des logements en
location courte durée à Bruxelles. Vous avez les 6 721 annonces Airbnb de la
ville, telles qu'elles étaient le 21 juin 2025.

Remarquez la ligne ci-dessous : on lit un fichier depuis une **adresse web**,
pas depuis un fichier local. Aucune donnée n'est stockée dans le dépôt du cours.
Le chapitre 2 explique pourquoi, en détail.

In [ ]:
import pandas as pd

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"

annonces = pd.read_csv(f"{BASE}/airbnb_bruxelles_raw.csv", low_memory=False)

print(f"{annonces.shape[0]} annonces, {annonces.shape[1]} colonnes")
annonces.head(3)

Regardons ce que pandas pense de ces colonnes.

In [ ]:
annonces.info()

Prenez trente secondes pour lire cette sortie. Deux choses devraient vous
surprendre.

D'abord, `price` est de type `object` (ou `str`, selon la version de pandas),
c'est-à-dire du texte. Regardons.

In [ ]:
annonces[["price", "host_response_rate", "host_since", "host_is_superhost"]].head()

Voilà le vrai visage des données.

- `price` vaut `$64.00` : un prix stocké en texte, avec un symbole monétaire.
  Impossible d'en faire une moyenne en l'état.
- `host_response_rate` vaut `100%` : un pourcentage en texte.
- `host_since` vaut `2010-08-11` : une date en texte, que pandas ne sait pas
  ordonner correctement.
- `host_is_superhost` vaut `t` : un booléen déguisé.

Ce n'est pas un défaut de ce jeu de données en particulier. **C'est l'état
normal des données réelles.** Le chapitre 3 est entièrement consacré à cela.

### Hack-Time 🛠️

Vous venez de voir que `annonces.info()` révèle des problèmes de type, et que
`annonces.head()` révèle des problèmes de contenu.

À votre tour, dans la cellule ci-dessous :

1. Affichez le taux de valeurs manquantes de chaque colonne, trié du plus
   manquant au moins manquant. Une piste : `annonces.isna().mean()` donne la
   proportion, et `.sort_values(ascending=False)` la trie.
2. Repérez **la colonne qui ne contient aucune information du tout**.
3. Dites à quelle étape du cycle de vie vous êtes en train de travailler.

**Cinq minutes.**

**Défi optionnel**
Les colonnes `review_scores_rating`, `reviews_per_month` et `first_review` ont
exactement le même taux de manquants. Ce n'est pas un hasard. Quelle est
l'explication, et qu'est-ce que cela change pour la suite ?

In [ ]:
# Votre code ici

## 1.3. Reconnaître un projet qui échoue

Voici une histoire vraie, résumée.

> Une chaîne de magasins veut prédire les ruptures de stock. L'équipe data
> récupère deux ans d'historique de ventes, construit un modèle qui atteint 94 %
> de précision, et le présente à la direction. Le projet est abandonné six
> semaines plus tard.
>
> En production, le modèle annonce les ruptures **le jour même**. Les
> réapprovisionnements prennent onze jours. La prédiction est juste, et elle
> n'est utilisable par personne.

Le modèle n'avait aucun défaut technique. Le projet a échoué à l'étape 1 :
personne n'avait demandé *à quel horizon une prédiction devient actionnable*.
Et cette erreur est restée invisible jusqu'à la mise en production, parce que
toutes les étapes suivantes ont été faites correctement.

C'est le schéma le plus fréquent des projets ratés : une erreur au début, qui ne
se voit qu'à la fin.

### Hack-Time 🛠️

Trois projets, trois échecs. Pour chacun, identifiez **l'étape du cycle de vie**
qui a été bâclée, et dites ce qu'il aurait fallu faire à ce moment-là.

1. Un modèle de détection de fraude atteint 99,8 % d'accuracy. Mis en production,
   il ne détecte aucune fraude. *(Indice : 0,2 % des transactions sont
   frauduleuses.)*
2. Un modèle de prévision des ventes est excellent sur les données de test.
   Trois mois après sa mise en service, ses prédictions sont devenues absurdes.
3. Une équipe livre une segmentation client en onze groupes, avec une analyse de
   quarante pages. Six mois plus tard, le marketing n'a rien changé à ses
   campagnes.

**Cinq minutes**, puis mise en commun.

**Défi optionnel**
Le cas 1 sera traité au chapitre 7, le cas 2 au chapitre 8, le cas 3 au
chapitre 11. Lequel des trois vous paraît le plus difficile à éviter, et
pourquoi ?

*Vos réponses :*

1.
2.
3.

## 1.4. Les trois paradigmes, en pratique

Data Analytics vous a présenté les trois familles. Un rappel en trois lignes,
puis l'exercice : la difficulté n'est pas de réciter les définitions, c'est de
reconnaître le paradigme derrière une demande formulée en langage métier.

- **Supervisé** : on dispose d'exemples où **la réponse est connue**. Chapitres
  5 à 10. Deux sous-familles, selon la nature de la réponse :

| | Ce qu'on prédit | Exemple chez nous | Exemples ailleurs |
|---|---|---|---|
| **Régression** | Un nombre | Le prix d'une annonce | Chiffre d'affaires, délai de livraison, consommation |
| **Classification** | Une catégorie | L'annonce est-elle Superhost | Client qui part ou non, mail spam ou non, défaut de paiement |

- **Non supervisé** : **aucune réponse** connue, on cherche une structure.
  Personne ne peut dire si le résultat est *juste*, seulement s'il est
  *utile* : c'est ce qui le rend difficile à évaluer. Chapitres 11 et 12.
- **Par renforcement** : un agent apprend en agissant et en recevant des
  récompenses. Il faut pouvoir expérimenter en vrai, ce qui est rare en
  gestion. Nous ne le traiterons pas.

La distinction régression ou classification n'est pas cosmétique : elle change
les modèles utilisables et, surtout, la façon de mesurer si le modèle est bon.
C'est l'objet du chapitre 7.

### Hack-Time 🛠️

Six demandes. Pour chacune : **supervisé (régression), supervisé
(classification), non supervisé, ou renforcement** ?

1. Estimer la durée de traitement d'une commande à partir de son contenu.
2. Regrouper 40 000 clients en profils d'achat comparables, sans idée préalable
   des profils.
3. Détecter si un paiement est frauduleux, à partir de 200 000 paiements dont on
   sait lesquels l'étaient.
4. Ajuster le prix d'un billet d'avion heure par heure pour maximiser le revenu
   du vol.
5. Repérer les 50 transactions les plus atypiques du mois, sans liste de fraudes
   connues.
6. Prévoir le montant qu'un client dépensera l'année prochaine.

**Cinq minutes.** Attention, deux d'entre elles se ressemblent beaucoup et ne
relèvent pas du même paradigme.

**Défi optionnel**
Comparez les cas 3 et 5. Ils portent tous les deux sur des transactions
suspectes. Qu'est-ce qui, concrètement, fait basculer l'un dans le supervisé et
l'autre dans le non supervisé ? Et si demain votre entreprise commence à
étiqueter ses fraudes, dans quel sens le problème bascule-t-il ?

*Vos réponses :*

| # | Paradigme | Pourquoi |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |

## 1.5. Présentation du projet fil rouge

**Le contexte.** Vous êtes data scientist dans une société de gestion de
logements en location courte durée à Bruxelles. Votre interlocutrice est la
responsable du revenue management. Elle vous pose deux questions, et ce sont
elles que nous allons traiter jusqu'à la fin du cours.

> **Question 1.** « Quand un propriétaire nous confie un nouveau logement, à quel
> prix par nuit faut-il le mettre en ligne ? »

Un prix, donc un nombre : c'est un problème de **régression**. La cible s'appelle
`y_price` dans nos données.

> **Question 2.** « Parmi les logements que nous gérons, lesquels ont le profil
> pour décrocher le statut Superhost ? On voudrait concentrer nos efforts
> d'accompagnement sur ceux-là. »

Superhost ou pas : c'est un problème de **classification**. La cible s'appelle
`y_superhost`.

Un même jeu de données, deux questions de nature différente. C'est volontaire :
vous verrez que presque tout change entre les deux, y compris des choses qui ne
sont pas évidentes, comme la façon de mesurer la performance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"

train = pd.read_csv(f"{BASE}/airbnb_bruxelles_train.csv")
# La table de modélisation, déjà nettoyée : elle sert aux chapitres 5 à 12.
print(f"{train.shape[0]} annonces, {train.shape[1]} colonnes")

In [ ]:
# Question 1 : le prix. Une variable continue.
print(train["y_price"].describe().round(1).to_string())

In [ ]:
# Question 2 : le statut Superhost. Une variable binaire.
repartition = train["y_superhost"].value_counts(normalize=True).sort_index()
print(repartition.rename({0: "Pas Superhost", 1: "Superhost"}).round(3).to_string())

Deux chiffres à retenir dès maintenant, parce qu'ils reviendront.

**Le prix médian est de 88 euros, la moyenne de 107.** La moyenne est nettement
au-dessus de la médiane : la distribution est étirée vers la droite par quelques
logements chers. Regardons.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(train["y_price"], bins=60, color="#4C72B0", edgecolor="white")
axes[0].axvline(train["y_price"].median(), color="#C44E52", linestyle="--", label="médiane")
axes[0].axvline(train["y_price"].mean(), color="#DD8452", linestyle="--", label="moyenne")
axes[0].set_title("Prix par nuit")
axes[0].set_xlabel("euros")
axes[0].legend()

axes[1].bar(["Pas Superhost", "Superhost"], repartition.values, color=["#CCCCCC", "#4C72B0"])
axes[1].set_title("Statut Superhost")
axes[1].set_ylabel("proportion")

plt.tight_layout()
plt.show()

**Seulement 27,6 % des annonces sont Superhost.** Les deux classes ne sont pas
équilibrées. Gardez ce chiffre en tête : au chapitre 7, il expliquera pourquoi un
modèle qui répond systématiquement « pas Superhost » obtient 72 % de bonnes
réponses tout en étant strictement inutile.

### Hack-Time 🛠️

On vient de décrire les deux cibles. À votre tour d'aller chercher un premier
élément de réponse métier, dans la cellule ci-dessous.

1. Calculez le **prix médian par type de logement** (`room_type`).
   Une piste : `train.groupby("room_type")["y_price"].median()`.
2. Ajoutez le **nombre d'annonces** de chaque type, pour savoir si vous pouvez
   faire confiance à ces médianes. `.agg(["median", "count"])` fait les deux
   d'un coup.
3. En une phrase : que répondriez-vous à la responsable du revenue management
   si elle vous demandait maintenant le prix d'un `Private room` ?

**Cinq minutes.**

**Défi optionnel**
Refaites le calcul par commune (`neighbourhood_cleansed`) et triez par prix
médian décroissant. La commune la plus chère vous surprend-elle ? Regardez son
nombre d'annonces avant de conclure quoi que ce soit.

In [ ]:
# Votre code ici

## À retenir

- Ce cours met en pratique ce que Data Analytics a posé : on code, les données
  sont sales, et le travail doit être reproductible.
- Six étapes : comprendre le problème, explorer, préparer, modéliser, valider,
  communiquer. On y revient sans arrêt, et chaque devoir les parcourt toutes.
- Les données réelles arrivent sales. Un prix qui vaut `$64.00` n'est pas un
  accident, c'est la norme.
- Dans le supervisé, prédire un **nombre** est une régression, prédire une
  **catégorie** est une classification. Cette distinction commande le choix du
  modèle et surtout la façon de l'évaluer.
- Notre fil rouge porte les deux : `y_price` en régression, `y_superhost` en
  classification, avec des classes déséquilibrées (27,6 % contre 72,4 %).

**La suite** : le chapitre 2 traite de la reproductibilité. Comment organiser un
projet, où vivent les données, et comment utiliser Git et GitHub. C'est le
chapitre qui vous servira dans tous les devoirs.